# LeetCode #1477: Find Two Non-overlapping Sub-arrays Each With Target Sum

https://leetcode.com/problems/find-two-non-overlapping-sub-arrays-each-with-target-sum/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(1)$ |
| **Optimal: Prefix Sum + DP ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Enumerate all pairs of non-overlapping subarrays that each sum to `target` and track the minimum combined length. $O(n^2)$ per subarray pair.

### Optimal: Prefix Sum + DP ★
Use a hashmap of prefix sums to find in $O(1)$ whether a subarray ending at `i` sums to `target`. Keep a running `bestLeft[i]` = shortest valid subarray ending at or before index `i`. A second pass from left builds `bestRight[i]` similarly. The answer is the minimum of `bestLeft[i-1] + bestRight[i]` for all split points `i`.

**Constraints:**
* `1 <= arr.length <= 10^5`
* `1 <= arr[i] <= 1000`
* `1 <= target <= 10^8`

## Solutions

### C#

In [ ]:
public class Solution {
    public int MinSumOfLengths(int[] arr, int target) {
        int n = arr.Length;
        int[] best = new int[n];   // best[i] = min subarray length ending at or before i
        int INF = int.MaxValue / 2;
        for (int i = 0; i < n; i++) best[i] = INF;

        // Prefix sum hashmap: prefixSum → index
        var prefixIdx = new Dictionary<int, int> {{0, -1}};
        int prefix = 0, minLen = INF, ans = INF;

        for (int i = 0; i < n; i++) {
            prefix += arr[i];
            // Check if a subarray ending at i sums to target
            if (prefixIdx.TryGetValue(prefix - target, out int j)) {
                int len = i - j;
                // If there's a valid left subarray, update the answer
                if (j > 0 && best[j - 1] < INF)
                    ans = Math.Min(ans, len + best[j - 1]);
                minLen = Math.Min(minLen, len);
            }
            best[i] = minLen;
            prefixIdx[prefix] = i;
        }
        return ans == INF ? -1 : ans;
    }
}

### Python

In [ ]:
class Solution:
    def minSumOfLengths(self, arr: list[int], target: int) -> int:
        n = len(arr)
        INF = float('inf')
        best = [INF] * n   # best[i] = min subarray length ending at or before i

        prefix_idx = {0: -1}
        prefix = 0
        min_len = INF
        ans = INF

        for i, v in enumerate(arr):
            prefix += v
            # Does a subarray ending at i sum to target?
            if (prefix - target) in prefix_idx:
                j = prefix_idx[prefix - target]
                length = i - j
                # Combine with the shortest valid subarray ending before this one starts
                if j > 0 and best[j - 1] < INF:
                    ans = min(ans, length + best[j - 1])
                min_len = min(min_len, length)
            best[i] = min_len
            prefix_idx[prefix] = i

        return -1 if ans == INF else ans

### Go

In [ ]:
func minSumOfLengths(arr []int, target int) int {
    n := len(arr)
    const INF = 1<<30
    best := make([]int, n)
    for i := range best { best[i] = INF }

    prefixIdx := map[int]int{0: -1}
    prefix, minLen, ans := 0, INF, INF

    for i, v := range arr {
        prefix += v
        // A subarray ending at i sums to target if prefix-target was seen before
        if j, ok := prefixIdx[prefix-target]; ok {
            length := i - j
            // Pair with the best non-overlapping left subarray
            if j > 0 && best[j-1] < INF {
                if v := length + best[j-1]; v < ans { ans = v }
            }
            if length < minLen { minLen = length }
        }
        best[i] = minLen
        prefixIdx[prefix] = i
    }
    if ans == INF { return -1 }
    return ans
}

### Rust

In [ ]:
use std::collections::HashMap;

impl Solution {
    pub fn min_sum_of_lengths(arr: Vec<i32>, target: i32) -> i32 {
        let n = arr.len();
        let inf = i32::MAX / 2;
        let mut best = vec![inf; n];
        let mut prefix_idx: HashMap<i32, i32> = HashMap::new();
        prefix_idx.insert(0, -1);
        let mut prefix = 0i32;
        let mut min_len = inf;
        let mut ans = inf;

        for (i, &v) in arr.iter().enumerate() {
            prefix += v;
            // Check if a subarray ending at i sums to target
            if let Some(&j) = prefix_idx.get(&(prefix - target)) {
                let length = i as i32 - j;
                // Combine with shortest valid non-overlapping left subarray
                if j > 0 && best[(j - 1) as usize] < inf {
                    ans = ans.min(length + best[(j - 1) as usize]);
                }
                min_len = min_len.min(length);
            }
            best[i] = min_len;
            prefix_idx.insert(prefix, i as i32);
        }
        if ans >= inf { -1 } else { ans }
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `arr = [3,2,2,4,3], target = 3`
Subarrays summing to 3: `[3]`(len 1, ends at 0) and `[3]`(len 1, ends at 4). Non-overlapping pair: indices 0 and 4. Total length: **2**.

### 2. Slightly Complex
**Input:** `arr = [7,3,4,7], target = 7`
`[7]` at index 0 (len 1) and `[7]` at index 3 (len 1). `best[2] = 1`; at i=3, ans = `1 + 1 = 2`. Answer: **2**.

### 3. Edge Case: Time Factor
**Input:** $n = 10^5$ elements all equal to 1, `target = 50000`.
Each prefix sum lookup is $O(1)$; the loop runs $n$ times — $O(n)$ total.

### 4. Edge Case: Space Factor
**Input:** $n = 10^5$ distinct prefix sums.
`prefix_idx` holds up to $n$ entries, `best` array length $n$ — $O(n)$ space.

### 5. Almost-Impossible but Plausible
**Input:** `arr = [1000]*100, target = 100000`
Sum of the entire array is $10^5$. Only one subarray (the whole array) sums to target; no room for a second non-overlapping one. Answer: **-1**.